# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "comparison_equation_inactive_months"
RAW_EQ = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [ ]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(80, 11), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=24, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=24)
    axs[0].tick_params(axis='both', which='major', labelsize=20)
    axs[0].tick_params(axis='both', which='minor', labelsize=18)
    axs[0].legend(fontsize = 18)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=20)
    axs[0].set_ylabel("Dst (nT)", fontsize=20)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")
    axs[1].set_title("Prediction Error", fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)
    axs[1].legend(fontsize=18)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=20)
    axs[1].tick_params(axis='both', which='minor', labelsize=18)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
        fontsize = 20,
        plot_legend=True
    )

    plt.savefig(save_path)
    plt.close()


In [5]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Inactive Months

In [6]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
inactive_months = storm_dates.INACTIVE_MONTHS
storm_indices = []
for sd, ed, storm_id in tqdm(inactive_months):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:05<00:00,  1.21it/s]


In [7]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,1.0,9.065700,7.193330,0.211086,0.666470,11.932473,9.006677,6.709943,0.221325,0.700900,...,8.783349,6.261033,0.259462,0.573727,13.281285,8.442873,5.885975,0.315762,0.620398,12.344069
1,2.0,10.327550,9.089918,0.078933,0.649493,11.227553,9.252288,6.887241,0.260744,0.698361,...,8.461255,6.338755,0.381747,0.635040,10.852909,8.398768,6.112439,0.390845,0.640084,10.740515
2,3.0,9.177884,7.077648,0.451978,0.709354,13.427162,10.253668,7.945776,0.315976,0.643944,...,10.155321,7.569446,0.329035,0.593816,17.366639,10.440272,7.841260,0.290853,0.559330,18.401390
3,4.0,8.923051,7.056695,0.097884,0.625979,14.002180,6.644148,5.343465,0.499834,0.719899,...,8.978698,6.886872,0.086598,0.574169,13.588300,9.031355,6.908010,0.075853,0.556463,14.567392
4,5.0,8.519445,6.608213,0.182974,0.716688,8.966265,7.652067,5.900868,0.340870,0.669356,...,7.779874,6.303559,0.318668,0.706829,9.523403,7.612389,6.218497,0.347688,0.706667,9.511693
5,6.0,7.932008,6.216470,0.586285,0.835410,7.744164,7.980740,6.022197,0.581186,0.839918,...,7.846606,6.101074,0.595146,0.782913,7.980364,7.441149,5.843710,0.635905,0.811793,8.960437
6,7.0,9.809770,7.417657,0.759212,0.901396,10.513068,17.133028,12.609196,0.265510,0.757418,...,11.903613,9.659758,0.645452,0.830475,12.021055,11.495900,9.293858,0.669323,0.849830,12.755869
7,Mean,9.107915,7.237133,0.338336,0.729256,11.116124,9.703231,7.345527,0.355064,0.718542,...,9.129816,7.017214,0.373730,0.670995,12.087708,8.980387,6.871964,0.389461,0.677795,12.468766
8,Global,9.136301,7.236310,0.535876,0.794786,14.179968,10.215139,7.342013,0.419795,0.724561,...,9.237484,7.017996,0.525539,0.730317,14.547831,9.095454,6.873437,0.540017,0.737866,15.643169


In [8]:
display(summary_df.iloc[-2:, :].style.format({col: "{:.2f}" for col in columns}))

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,Burton_BFE,OBM_RMSE,OBM_MAE,OBM_R2,OBM_CC,OBM_BFE,DDM1_RMSE,DDM1_MAE,DDM1_R2,DDM1_CC,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
7,Mean,9.11,7.24,0.34,0.73,11.12,9.70,7.35,0.36,0.72,11.39,8.74,6.76,0.39,0.78,10.83,9.00,6.96,0.39,0.68,12.36,9.13,7.02,0.37,0.67,12.09,8.98,6.87,0.39,0.68,12.47
8,Global,9.14,7.24,0.54,0.79,14.18,10.22,7.34,0.42,0.72,22.07,8.84,6.74,0.57,0.81,16.25,9.12,6.96,0.54,0.73,14.99,9.24,7.02,0.53,0.73,14.55,9.10,6.87,0.54,0.74,15.64


In [9]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,1.0,11.932473,12.412932,12.009661,13.198517,13.281285,12.344069
1,2.0,11.227553,7.291708,9.174432,10.743051,10.852909,10.740515
2,3.0,13.427162,11.004200,12.158554,17.635082,17.366639,18.401390
3,4.0,14.002180,7.934079,10.873798,14.246018,13.588300,14.567392
4,5.0,8.966265,8.331058,9.987242,9.522711,9.523403,9.511693
5,6.0,7.744164,7.952329,6.250693,8.662262,7.980364,8.960437
6,7.0,10.513068,24.788263,15.355667,12.542709,12.021055,12.755869
7,Mean,11.116124,11.387795,10.830007,12.364336,12.087708,12.468766
8,Global,14.179968,22.071986,16.246798,14.991335,14.547831,15.643169


In [10]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm Index & Equation_RMSE & Equation_MAE & Equation_R2 & Equation_CC & Equation_BFE & Burton_RMSE & Burton_MAE & Burton_R2 & Burton_CC & Burton_BFE & OBM_RMSE & OBM_MAE & OBM_R2 & OBM_CC & OBM_BFE & DDM1_RMSE & DDM1_MAE & DDM1_R2 & DDM1_CC & DDM1_BFE & DDM2_RMSE & DDM2_MAE & DDM2_R2 & DDM2_CC & DDM2_BFE & DDM3_RMSE & DDM3_MAE & DDM3_R2 & DDM3_CC & DDM3_BFE \\
\midrule
Mean & 9.108 & 7.237 & 0.338 & 0.729 & 11.116 & 9.703 & 7.346 & 0.355 & 0.719 & 11.388 & 8.739 & 6.760 & 0.391 & 0.781 & 10.830 & 8.998 & 6.959 & 0.392 & 0.675 & 12.364 & 9.130 & 7.017 & 0.374 & 0.671 & 12.088 & 8.980 & 6.872 & 0.389 & 0.678 & 12.469 \\
\bottomrule
\end{tabular}



In [11]:
print(summary_df)

  Storm Index  Equation_RMSE  Equation_MAE  Equation_R2  Equation_CC  \
0         1.0       9.065700      7.193330     0.211086     0.666470   
1         2.0      10.327550      9.089918     0.078933     0.649493   
2         3.0       9.177884      7.077648     0.451978     0.709354   
3         4.0       8.923051      7.056695     0.097884     0.625979   
4         5.0       8.519445      6.608213     0.182974     0.716688   
5         6.0       7.932008      6.216470     0.586285     0.835410   
6         7.0       9.809770      7.417657     0.759212     0.901396   
7        Mean       9.107915      7.237133     0.338336     0.729256   
8      Global       9.136301      7.236310     0.535876     0.794786   

   Equation_BFE  Burton_RMSE  Burton_MAE  Burton_R2  Burton_CC  ...  \
0     11.932473     9.006677    6.709943   0.221325   0.700900  ...   
1     11.227553     9.252288    6.887241   0.260744   0.698361  ...   
2     13.427162    10.253668    7.945776   0.315976   0.643944  ..

In [12]:
df = summary_df.copy()

In [13]:
# Keep only the 20 individual storms
storm_df = df[
    ~df["Storm Index"].isin(["Mean", "Global"])
].copy()

baselines = {
    "Burton": "Burton",
    "OBM": "OBM",
    "DDM#1": "DDM1",
    "DDM#2": "DDM2",
    "DDM#3": "DDM3",
}

metrics = {
    "RMSE": "lower",
    "MAE": "lower",
    "R2": "higher",
    "CC": "higher",
    "BFE": "lower",
}

rows = []

for metric, direction in metrics.items():

    row = {
        "Metric": metric,
        "Equation": storm_df[f"Equation_{metric}"].mean(),
    }

    for baseline_name, prefix in baselines.items():

        eq = storm_df[f"Equation_{metric}"]
        baseline = storm_df[f"{prefix}_{metric}"]

        row[baseline_name] = baseline.mean()

        if direction == "lower":
            wins = (eq < baseline).sum()
        else:
            wins = (eq > baseline).sum()

        row[f"Wins vs {baseline_name}"] = wins

    rows.append(row)

results_df = pd.DataFrame(rows)

print(results_df)

  Metric   Equation     Burton  Wins vs Burton        OBM  Wins vs OBM  \
0   RMSE   9.107915   9.703231               3   8.738592            4   
1    MAE   7.237133   7.345527               2   6.759521            3   
2     R2   0.338336   0.355064               3   0.390764            4   
3     CC   0.729256   0.718542               3   0.780556            1   
4    BFE  11.116124  11.387795               3  10.830007            3   

       DDM#1  Wins vs DDM#1      DDM#2  Wins vs DDM#2      DDM#3  \
0   8.997886              3   9.129816              3   8.980387   
1   6.959147              2   7.017214              2   6.871964   
2   0.391694              3   0.373730              3   0.389461   
3   0.675370              6   0.670995              7   0.677795   
4  12.364336              6  12.087708              5  12.468766   

   Wins vs DDM#3  
0              3  
1              2  
2              3  
3              7  
4              6  


In [14]:
latex_table = results_df.to_latex(
    index=False,
    escape=False,
    float_format="%.2f"
)

print(latex_table)

\begin{tabular}{lrrrrrrrrrrr}
\toprule
Metric & Equation & Burton & Wins vs Burton & OBM & Wins vs OBM & DDM#1 & Wins vs DDM#1 & DDM#2 & Wins vs DDM#2 & DDM#3 & Wins vs DDM#3 \\
\midrule
RMSE & 9.11 & 9.70 & 3 & 8.74 & 4 & 9.00 & 3 & 9.13 & 3 & 8.98 & 3 \\
MAE & 7.24 & 7.35 & 2 & 6.76 & 3 & 6.96 & 2 & 7.02 & 2 & 6.87 & 2 \\
R2 & 0.34 & 0.36 & 3 & 0.39 & 4 & 0.39 & 3 & 0.37 & 3 & 0.39 & 3 \\
CC & 0.73 & 0.72 & 3 & 0.78 & 1 & 0.68 & 6 & 0.67 & 7 & 0.68 & 7 \\
BFE & 11.12 & 11.39 & 3 & 10.83 & 3 & 12.36 & 6 & 12.09 & 5 & 12.47 & 6 \\
\bottomrule
\end{tabular}



In [15]:
results_df

,Metric,Equation,Burton,Wins vs Burton,OBM,Wins vs OBM,DDM#1,Wins vs DDM#1,DDM#2,Wins vs DDM#2,DDM#3,Wins vs DDM#3
0,RMSE,9.107915,9.703231,3,8.738592,4,8.997886,3,9.129816,3,8.980387,3
1,MAE,7.237133,7.345527,2,6.759521,3,6.959147,2,7.017214,2,6.871964,2
2,R2,0.338336,0.355064,3,0.390764,4,0.391694,3,0.373730,3,0.389461,3
3,CC,0.729256,0.718542,3,0.780556,1,0.675370,6,0.670995,7,0.677795,7
4,BFE,11.116124,11.387795,3,10.830007,3,12.364336,6,12.087708,5,12.468766,6
